# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [19]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv') 
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

clustering
Cluster profiles: for each cluster found, a description of its typical numbers — e.g. average impressions, CTR, position, word count, freshness, trend — plus how many pages fall into it.

In [ ]:
# No forward-looking label exists — trend_direction is a current-window bucket, not a future outcome
print(df['trend_direction'].value_counts())
print("\nNo product decision fields present (confirms observable-signals-only data):")
product_flags = ['health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win']
print([c for c in product_flags if c in df.columns])  # should print []

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

No product decision fields present (confirms observable-signals-only data):
[]


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
*since clustering doesn't have a target/label at all, the concept of "proxy" doesn't really apply*

In [ ]:
# Candidate features for clustering — numeric, observable signals only
cluster_features = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days',
    'days_since_last_update', 'trend_pct'
]
print(df[cluster_features].isnull().sum())
print(df[cluster_features].describe())

impressions_90d              0
clicks_90d                   0
sessions_90d                 0
ctr                          0
avg_position                 0
engagement_rate              0
scroll_rate                125
word_count                7699
content_age_days             0
days_since_last_update       0
trend_pct                 3388
dtype: int64
       impressions_90d    clicks_90d  sessions_90d           ctr  \
count     30000.000000  30000.000000  30000.000000  30000.000000   
mean       5200.366300     16.097333     37.066633      0.510733   
std       16838.019547     75.076958    107.069131      3.279162   
min           1.000000      0.000000      1.000000      0.000000   
25%          81.000000      0.000000      2.000000      0.000000   
50%         731.000000      1.000000      7.000000      0.070000   
75%        3615.250000      7.000000     27.000000      0.290000   
max      517715.000000   4178.000000   4345.000000    100.000000   

       avg_position  engagement_r

## 3. Success metric

*One metric you can defend. What number means 'good'?*
*"Good" = high agreement (ARI close to 1) between cluster assignments across repeated runs — meaning the archetypes are a real, reproducible pattern in the data, not an artifact of one random seed. Low agreement means the clusters can't be trusted as a decision-support lens yet.*

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from sklearn.impute import SimpleImputer

X = SimpleImputer(strategy='median').fit_transform(df[cluster_features])
X_scaled = StandardScaler().fit_transform(X)

# Run K-Means twice with different seeds, compare assignments
labels_a = KMeans(n_clusters=6, random_state=1, n_init=10).fit_predict(X_scaled)
labels_b = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(X_scaled)

ari = adjusted_rand_score(labels_a, labels_b)
print(f"Adjusted Rand Index between seeds: {ari:.3f}")
# Close to 1 = stable/reproducible clusters. Near 0 = clusters are noise, not structure.

Adjusted Rand Index between seeds: 0.965


In [ ]:
# Confirm no raw private fields exist anywhere in the columns
forbidden = ['url', 'query', 'title', 'domain', 'client_name', 'keyword_text']
leaked = [c for c in df.columns if any(f in c.lower() for f in forbidden)]
print("Flagged columns (should be empty):", leaked)

Flagged columns (should be empty): []


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
*One row = one content item (one page), identified by content_id. This is the dim_content grain in the warehouse, and in the starter dataset every row is already deduplicated to one page — confirmed below by checking content_id is unique across all 30,000 rows. This is the correct grain for archetype clustering: I want to group pages by how they behave, not days or events.*

In [ ]:


print("Total rows:", len(df))
print("Unique content_id:", df['content_id'].nunique())
print("One row per content item?", len(df) == df['content_id'].nunique())

# Show the slice
df.head(5)

Total rows: 30000
Unique content_id: 30000
One row per content item? True


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
*A fixed rule (e.g. "if impressions_90d > 500 and ctr < 0.5, call it archetype X") can only look at one or two conditions at a time, chosen in advance. But an archetype is really a combination of many correlated signals — impressions, position, CTR, engagement, freshness, word count — moving together in ways a human wouldn't reliably hand-pick thresholds for across 30,000 pages and 32 clients with very different baselines. A rule also can't adapt its boundaries to where the actual data clusters; it forces a boundary wherever the author guessed one. Clustering finds groups from the shape of the multi-dimensional data itself, which is exactly the kind of "too many interacting signals to write as one if-statement" case this lane is built for.*

In [ ]:
# Show that no single feature cleanly separates the data into obvious groups
print(df[['impressions_90d','ctr','avg_position','engagement_rate']].corr())
# Correlations are moderate/mixed, not near +1/-1 — no single variable acts as a stand-in
# for the others, so a one-variable if-statement can't capture the joint pattern.

                 impressions_90d       ctr  avg_position  engagement_rate
impressions_90d         1.000000 -0.018950     -0.070786         0.024318
ctr                    -0.018950  1.000000     -0.072590         0.096903
avg_position           -0.070786 -0.072590      1.000000        -0.017926
engagement_rate         0.024318  0.096903     -0.017926         1.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.